# Set 06 – Bias-Varianz-Trade-off mit Polynomen

Ein Modell soll den allgemeinen Zusammenhang lernen, nicht einzelne Trainingspunkte auswendig. Mit dem Regler verändern wir den Polynomgrad und damit die Modellkomplexität:

- kleiner Grad: wenig flexibel, hoher Bias, mögliches Underfitting
- mittlerer Grad: passende Balance
- großer Grad: sehr flexibel, hohe Varianz, mögliches Overfitting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import IntSlider, interact
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(42)

## 1. Kontrollierte Daten

Die wahre Funktion ist ein Polynom dritten Grades. Die Trainingswerte enthalten Rauschen. Für die Visualisierung kennen wir ausnahmsweise auch die rauschfreie Funktion.

In [ ]:
def wahre_funktion(x):
    return 0.55 * x**3 - 1.2 * x**2 - 0.8 * x + 5

n_train = 28
x_train = np.sort(rng.uniform(-3.2, 3.2, n_train))
y_train = wahre_funktion(x_train) + rng.normal(0, 2.2, n_train)

n_validierung = 220
x_validierung = rng.uniform(-3.2, 3.2, n_validierung)
y_validierung = wahre_funktion(x_validierung) + rng.normal(0, 2.2, n_validierung)

X_train = pd.DataFrame({"x": x_train})
X_validierung = pd.DataFrame({"x": x_validierung})
x_gitter = np.linspace(-3.4, 3.4, 500)
X_gitter = pd.DataFrame({"x": x_gitter})

## 2. Polynom-Pipeline

PolynomialFeatures erzeugt x, x², x³ und weitere Potenzen. StandardScaler bringt diese Spalten auf vergleichbare Skalen. LinearRegression lernt anschließend ihre Gewichte.

In [ ]:
def erstelle_modell(grad):
    return Pipeline([
        ("polynom", PolynomialFeatures(degree=grad, include_bias=False)),
        ("skalierung", StandardScaler()),
        ("modell", LinearRegression()),
    ])

## 3. Interaktiver Polynomgrad

Bewege den Regler von 1 bis 18. Beobachte dabei:

- Grad 1 trifft die Kurve systematisch nicht.
- Ein mittlerer Grad beschreibt den Grundverlauf.
- Hohe Grade können zwischen den wenigen Trainingspunkten stark schwingen.
- Der Trainingsfehler sinkt meist weiter, während der Validierungsfehler wieder steigen kann.

In [ ]:
@interact(
    grad=IntSlider(value=3, min=1, max=18, step=1, description="Polynomgrad")
)
def zeige_polynom(grad):
    modell = erstelle_modell(grad)
    modell.fit(X_train, y_train)

    y_train_pred = modell.predict(X_train)
    y_val_pred = modell.predict(X_validierung)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    val_rmse = np.sqrt(mean_squared_error(y_validierung, y_val_pred))
    y_gitter_pred = modell.predict(X_gitter)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.scatter(x_train, y_train, color="#4C78A8", s=45, label="Training")
    ax.plot(x_gitter, wahre_funktion(x_gitter), color="black", linestyle="--", linewidth=2, label="wahre Funktion")
    ax.plot(x_gitter, y_gitter_pred, color="#E45756", linewidth=2.5, label=f"Modell mit Grad {grad}")
    ax.set_ylim(-35, 35)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Train-RMSE {train_rmse:.2f} | Validierungs-RMSE {val_rmse:.2f}")
    ax.legend()
    plt.show()

## 4. Fehlerkurven über alle Grade

Das beste Modell wird nicht am kleinsten Trainingsfehler gewählt. Gesucht ist der Grad mit dem kleinsten Validierungsfehler.

In [ ]:
ergebnisse = []
for grad in range(1, 19):
    modell = erstelle_modell(grad)
    modell.fit(X_train, y_train)
    ergebnisse.append({
        "Grad": grad,
        "Train_RMSE": np.sqrt(mean_squared_error(y_train, modell.predict(X_train))),
        "Validierungs_RMSE": np.sqrt(mean_squared_error(y_validierung, modell.predict(X_validierung))),
    })

fehler = pd.DataFrame(ergebnisse)
bester_grad = int(fehler.loc[fehler["Validierungs_RMSE"].idxmin(), "Grad"])
display(fehler.round(3))
print("Bester Grad nach Validierungsfehler:", bester_grad)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(fehler["Grad"], fehler["Train_RMSE"], marker="o", label="Training")
ax.plot(fehler["Grad"], fehler["Validierungs_RMSE"], marker="o", label="Validierung")
ax.axvline(bester_grad, color="gray", linestyle="--", label=f"bester Grad {bester_grad}")
ax.set_xticks(range(1, 19))
ax.set_xlabel("Polynomgrad")
ax.set_ylabel("RMSE")
ax.set_title("Trainings- und Validierungsfehler")
ax.legend()
plt.show()

## 5. Varianz sichtbar machen

Nun ziehen wir viele leicht unterschiedliche Trainingsstichproben. Der Regler steuert wieder den Grad. Das farbige Band zeigt, wie stark sich die Vorhersagen zwischen den Stichproben unterscheiden.

Ein breites Band bedeutet hohe Varianz.

In [ ]:
@interact(
    grad=IntSlider(value=3, min=1, max=15, step=1, description="Polynomgrad")
)
def zeige_varianz(grad):
    vorhersagen = []
    for seed in range(30):
        lokale_rng = np.random.default_rng(seed)
        x_stichprobe = np.sort(lokale_rng.uniform(-3.2, 3.2, n_train))
        y_stichprobe = wahre_funktion(x_stichprobe) + lokale_rng.normal(0, 2.2, n_train)
        X_stichprobe = pd.DataFrame({"x": x_stichprobe})

        modell = erstelle_modell(grad)
        modell.fit(X_stichprobe, y_stichprobe)
        vorhersagen.append(modell.predict(X_gitter))

    vorhersagen = np.asarray(vorhersagen)
    mittel = vorhersagen.mean(axis=0)
    standardabweichung = vorhersagen.std(axis=0)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(x_gitter, wahre_funktion(x_gitter), color="black", linestyle="--", label="wahre Funktion")
    ax.plot(x_gitter, mittel, color="#4C78A8", linewidth=2.5, label="mittlere Vorhersage")
    ax.fill_between(
        x_gitter,
        mittel - standardabweichung,
        mittel + standardabweichung,
        color="#4C78A8",
        alpha=0.25,
        label="plus/minus eine Standardabweichung",
    )
    ax.set_ylim(-35, 35)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Empfindlichkeit gegenüber Trainingsstichproben – Grad {grad}")
    ax.legend()
    plt.show()

## Zusammenfassung

- Underfitting: Train- und Validierungsfehler sind beide hoch.
- Gute Balance: beide Fehler sind niedrig und liegen relativ nah beieinander.
- Overfitting: Trainingsfehler ist sehr niedrig, Validierungsfehler und Varianz steigen.
- Der Polynomgrad ist ein Hyperparameter und wird mit Validierungsdaten oder Cross-Validation gewählt.
- Ein finaler Testsatz darf erst nach der Modellwahl verwendet werden.